# Experiment: SpectralBio Reject Recovery Accept Upgrade

This notebook builds the supplementary evidence bundle meant to move SpectralBio from reject-recovery mode toward Accept / Strong Accept.

## What this notebook does
- Runs a TP53 nested-CV leakage audit on the released `frob_dist + ll_proper` surface.
- Compares TP53 performance on the released ESM2-150M reference surface versus a stronger ESM2-650M backbone.
- Builds and scores a supplementary multigene panel beyond TP53 using local assets plus ClinVar/UniProt-backed additions.
- Writes machine-readable CSV/JSON outputs, publication-ready PNG figures, an experiment log, and a final zip bundle.

## Runtime guidance
- Nested CV leakage audit: usually minutes.
- TP53 650M rerun: commonly tens of minutes on Colab T4.
- Multigene panel: can range from under 1 hour to several hours depending on selected genes and whether checkpoints already exist.

## Safety / scope
- The canonical public benchmark contract is not modified here.
- All outputs are written under a supplementary reject-recovery directory.
- If a stronger backbone or extra gene weakens the current claim, keep the result and document it honestly.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioRejectRecovery')

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs will stay in the Colab VM unless you override OUTPUT_ROOT later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
REPO_ROOT = Path('/content/Stanford-Claw4s')

if not REPO_ROOT.exists():
    print('Cloning repo...')
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('Installing editable package and notebook extras...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', 'pandas', 'matplotlib'])

print('Repo ready at', REPO_ROOT)
print('Repo branch:', REPO_BRANCH)
print('Added to sys.path:', src_path)


## Plan

### Priority A
- Leakage audit with nested CV to answer the same-set alpha-tuning criticism directly.
- Stronger-backbone TP53 comparison to test whether covariance complementarity survives beyond ESM2-150M.
- Supplementary multigene panel to reduce the single-protein scope objection.

### Priority B
- Clean CSV/JSON/PNG exports for manuscript and release-surface updates.
- Final experiment log and zip bundle.

### Execution rule
Always finish Experiments 1-3 before enabling any optional robustness extras.


In [ ]:
import json
import platform
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd

from spectralbio.supplementary.reject_recovery import (
    BackboneEvaluationConfig,
    MultiGeneConfig,
    NestedCVConfig,
    build_multigene_panel_manifest,
    create_reject_recovery_paths,
    create_reject_recovery_zip,
    run_multigene_panel,
    run_tp53_backbone_comparison,
    run_tp53_nested_cv_audit,
    write_experiment_log,
)

NOTEBOOK_SEED = 42
OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'reject_recovery'
)

nested_config = NestedCVConfig(
    n_splits=5,
    n_repeats=5,
    alpha_step=0.05,
    random_seed=NOTEBOOK_SEED,
    render_figures=True,
)

backbone_config = BackboneEvaluationConfig(
    reference_model_name='facebook/esm2_t30_150M_UR50D',
    stronger_model_name='facebook/esm2_t33_650M_UR50D',
    window_radius=40,
    pair_alpha=0.55,
    random_seed=NOTEBOOK_SEED,
    checkpoint_every=10,
    render_figures=True,
    overwrite=False,
    reuse_frozen_tp53_reference=True,
    run_full_surface_alpha_sweep=True,
)

multigene_config = MultiGeneConfig(
    candidate_genes=('PTEN', 'MSH2', 'MLH1', 'CHEK2', 'PALB2', 'BRCA2', 'ATM'),
    model_name='facebook/esm2_t30_150M_UR50D',
    window_radius=40,
    pair_alpha=0.55,
    random_seed=NOTEBOOK_SEED,
    checkpoint_every=10,
    bootstrap_replicates=1000,
    min_total=40,
    min_per_class=10,
    max_additional_genes=4,
    render_figures=True,
    overwrite=False,
    reuse_frozen_tp53_reference=True,
)

paths = create_reject_recovery_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)


In [ ]:
import json
import subprocess
import sys

try:
    gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        check=False,
        capture_output=True,
        text=True,
    )
    gpu_summary = gpu_probe.stdout.strip() if gpu_probe.stdout.strip() else 'GPU probe unavailable'
except FileNotFoundError:
    gpu_summary = 'nvidia-smi not found'

environment_snapshot = {
    'python_version': sys.version.split()[0],
    'platform': platform.platform(),
    'gpu_summary': gpu_summary,
    'repo_root': str(REPO_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'nested_config': asdict(nested_config),
    'backbone_config': asdict(backbone_config),
    'multigene_config': asdict(multigene_config),
}

(paths.configs / 'notebook_environment.json').write_text(json.dumps(environment_snapshot, indent=2) + '
', encoding='utf-8')
environment_snapshot


## Experiment 1: TP53 Leakage Audit

Goal: test whether the reported `0.55 / 0.45` pair still helps when alpha is tuned out-of-fold instead of on the full TP53 surface.


In [ ]:
nested_summary = run_tp53_nested_cv_audit(paths, nested_config)
nested_summary


In [ ]:
fold_df = pd.read_csv(paths.leakage_audit / 'tp53_nested_cv_fold_results.csv')
alpha_df = pd.read_csv(paths.leakage_audit / 'tp53_nested_cv_alpha_distribution.csv')

print('Leakage-audit summary path:', paths.leakage_audit / 'tp53_nested_cv_summary.json')
display(fold_df.head())
display(alpha_df['chosen_alpha'].value_counts().sort_index())


## Experiment 2: Stronger Backbone Validation

Goal: compare the released ESM2-150M TP53 surface with a supplementary ESM2-650M TP53 rerun and see whether covariance still contributes incremental signal.


In [ ]:
backbone_summary = run_tp53_backbone_comparison(paths, backbone_config)
backbone_summary


In [ ]:
backbone_df = pd.read_csv(paths.backbone_strength / 'tp53_backbone_comparison.csv')
print('Backbone metrics path:', paths.backbone_strength / 'tp53_backbone_metrics.json')
display(backbone_df)


## Experiment 3: Supplementary Multigene Panel

Goal: expand the evidence surface beyond TP53 using local BRCA1 data plus additional ClinVar/UniProt-backed genes that pass a fixed viability rule.


In [ ]:
panel_manifest = build_multigene_panel_manifest(paths, multigene_config)
panel_manifest['selected_genes']


In [ ]:
multigene_summary = run_multigene_panel(paths, multigene_config, panel_manifest=panel_manifest)
multigene_summary


In [ ]:
multigene_df = pd.read_csv(paths.multigene / 'multigene_summary.csv')
gene_table_df = pd.read_csv(paths.multigene / 'multigene_gene_table.csv')
print('Multigene metrics path:', paths.multigene / 'multigene_metrics.json')
display(multigene_df)
display(gene_table_df)


## Optional Robustness Layer

Only enable extra robustness after the three critical experiments above are complete and their outputs are safely written.


In [ ]:
RUN_OPTIONAL_ROBUSTNESS = False
optional_notes = []

if RUN_OPTIONAL_ROBUSTNESS:
    optional_notes.append('Optional robustness block enabled; add extra analyses here after validating critical outputs.')
else:
    optional_notes.append('Optional robustness block skipped by default to prioritize the critical reject-recovery evidence path.')

optional_notes


## Bundle and Final Notes

This final block writes a human-readable experiment log and creates the zipped results bundle used for paper/repo/release updates.


In [ ]:
completed_experiments = []
skipped_experiments = []

if (paths.leakage_audit / 'tp53_nested_cv_summary.json').exists():
    completed_experiments.append('TP53 nested CV leakage audit')
else:
    skipped_experiments.append('TP53 nested CV leakage audit')

if (paths.backbone_strength / 'tp53_backbone_metrics.json').exists():
    completed_experiments.append('TP53 backbone comparison')
else:
    skipped_experiments.append('TP53 backbone comparison')

if (paths.multigene / 'multigene_metrics.json').exists():
    completed_experiments.append('Supplementary multigene panel')
else:
    skipped_experiments.append('Supplementary multigene panel')

zip_path = create_reject_recovery_zip(paths)
log_path = write_experiment_log(paths, completed_experiments, skipped_experiments, optional_notes, zip_path)

final_notes = {
    'completed_experiments': completed_experiments,
    'skipped_experiments': skipped_experiments,
    'output_paths': {
        'root': str(paths.root),
        'leakage_audit': str(paths.leakage_audit),
        'backbone_strength': str(paths.backbone_strength),
        'multigene': str(paths.multigene),
        'figures': str(paths.figures),
        'tables': str(paths.tables),
        'logs': str(paths.logs),
        'configs': str(paths.configs),
    },
    'log_path': str(log_path),
    'zip_path': str(zip_path),
}

print(json.dumps(final_notes, indent=2))


## Final Notes

The notebook ends by printing the completed experiments, skipped experiments, output paths, and final zip path.
Use that payload as the handoff surface for the next paper/release revision pass.
